# TP18.1 — Concevoir le socle de l’assistant NutriScope

**Notebook pédagogique à compléter — première PR, TP18 non clôturé.**

Énoncé original : [README_tp_18.md](../../../docs/tps/README_tp_18.md).

Objectif : préparer un assistant qui répond à partir de fiches vérifiables, sans attendre la substitution du TP16, le RAG ou l’application. On avance par petits exercices : réfléchir, compléter un TODO, vérifier, puis continuer. Les réponses et les décisions restent à construire par vous.

Ce support ne constitue ni un prototype terminé ni une démonstration exécutée sur un vrai modèle. Une seconde partie réunira les intégrations et les preuves manquantes. Aucun résultat de modèle ou de catalogue n’est prérempli.

## 1 — Délimiter cette première partie

| Travail | Faisable maintenant ? | Condition / suite |
|---|---|---|
| Responsabilités, architecture et garde-fous | Oui | Distinguer conçu et réellement branché |
| Contrat et validation de `chercher_produit` | Oui | Aucun modèle ni catalogue nécessaire pour valider les arguments |
| Recherche déterministe et adaptation des fiches | Oui, si catalogue disponible | Inspection du schéma réel avant le mapping |
| Prompt, routage des outils, boucle bornée | Oui | Préparer hors ligne ; vraie exécution séparée |
| Comparatif des modes d’accès et ADR proposée | Oui | Conclusion finale après une mesure réelle |
| Premier appel sur vrai modèle et catalogue | Conditionnel | Possible dès cette partie ; ne dépend pas de TP16 |
| Recommandations via moteur TP16 | Différé | Ne pas inventer une alternative en attendant |
| RAG, orchestration TP21, API TP23, application | Différé | Préserver le contrat imposé |
| Clôture du TP18 | Différée | Démonstration exécutée et décisions étayées |

**À faire :** noter précisément les prérequis qui vous manquent. Ne pas attribuer au TP14 ou au TP17 une dépendance de l’appel outillé : lire un Nutri-Score existant n’exige pas de le prédire ni de reconnaître une image.

## 2 — Écrire les comportements avant le code

Compléter pour chaque cas : données requises, action autorisée, réponse si information absente.

| Question | Outil / garde | Critère de réussite |
|---|---|---|
| « Quel est le Nutri-Score de ce produit ? » | À compléter | Grade issu de la fiche, ou absence explicite |
| « Compare ces deux produits » | À compléter | Deux identités confirmées, mêmes unités |
| « Pourquoi ce score ? » | À compléter | Distinguer observation et calcul exact non disponible |
| « Propose une alternative » | À compléter | Annoncer la limite actuelle sans inventer de recommandation |
| Produit inconnu / nom ambigu | À compléter | Signaler l’absence / demander une précision |
| Conseil médical / hors alimentation | À compléter | Refus adapté |

**Questions :** un nom identique suffit-il à identifier un produit ? Un grade permet-il de reconstruire exactement le calcul du score ? Pourquoi ne pas remplacer une valeur inconnue par zéro ?

## 3 — Vérifier l’environnement, sans lancer de traitement

Choisir le kernel Python du projet (Python 3.12 selon le README). Cette cellule utilise uniquement la bibliothèque standard ; elle ne contacte aucun service et ne modifie aucune donnée.

In [ ]:
import sys
import json
from pathlib import Path

depart = Path.cwd().resolve()
RACINE = next((p for p in (depart, *depart.parents)
               if (p / "src/storage/commun.py").is_file()
               and (p / "docs/tps/README_tp_18.md").is_file()), None)
if RACINE is None:
    raise RuntimeError("Ouvrir ce notebook depuis le dépôt NutriScope.")
print("Python :", sys.version.split()[0])
print("Interpréteur :", sys.executable)
print("Racine :", RACINE)
print("Catalogue TP12 présent :", (RACINE / "data/parquet/produits").is_dir())

## 4 — Concevoir les composants

Point de départ à discuter, **pas une description d’un assistant déjà implémenté** :

```mermaid
flowchart TD
    U[Question] --> G[Contrôle du périmètre]
    G --> O[Orchestrateur borné]
    O <--> M[Client du vrai modèle]
    O --> V[Validation du nom et des arguments]
    V --> C[chercher_produit]
    C --> A[Adaptateur du catalogue en lecture seule]
    A --> D[Catalogue réel TP12]
    C --> O
    O --> R[Réponse et traces vérifiables]
    F[Substitution TP16 - à venir] -.-> O
    K[RAG - à venir] -.-> O
    I[API et application - à venir] -.-> G
```

**À compléter :** responsabilité et interface de chaque composant ; lieu du refus ; budget d’appels ; gestion des erreurs ; données conservées dans les traces.

Un prompt exprime les règles, mais ne garantit pas leur respect. Le programme impose la liste d’outils et les budgets. Un filtre par mots-clés peut aider à l’apprentissage, mais ne couvre ni les paraphrases ni tous les cas ambigus. Prévoir des exemples de faux positifs et de faux négatifs.

Livrable à rédiger après l’exercice : `docs/architecture.md` (absent de ce checkout lors de la préparation). Indiquer pour chaque brique : conçue, implémentée, testée ou différée.

## 5 — Formaliser le contrat imposé

Relire l’énoncé : `code` est prioritaire sur `nom`, `limite` vaut 3 par défaut et appartient à [1, 5]. Aucun autre argument. Entrée incorrecte : `{"erreur": "explication"}`, sans exception propagée au modèle.

**Exercice :** écrire le schéma JSON de l’outil. Comment exprimer « au moins nom ou code » ? Pourquoi vérifier aussi côté Python si le modèle reçoit déjà un schéma ?

Conventions à décider explicitement : chaînes vides ou blanches refusées ; booléens refusés comme limites malgré `isinstance(True, int)` ; rejet des nombres décimaux et des objets JSON non dictionnaires. Le code-barres reste du texte, y compris ses zéros de tête.

In [ ]:
# TODO : compléter le schéma depuis le tableau de l’énoncé.
# Selon le fournisseur, l’enveloppe du schéma peut varier : vérifier sa documentation.
SCHEMA_ARGUMENTS = {
    "type": "object",
    "properties": {},
    "additionalProperties": False,
}

def valider_arguments(entree):
    """Retourner un dictionnaire normalisé ou {"erreur": ...}, jamais exécuter l’entrée."""
    # TODO 1 : accepter un objet ou décoder une chaîne avec json.loads, jamais eval.
    # TODO 2 : contrôler le type de l’objet, les clés, les types et les bornes.
    # TODO 3 : contrôler nom/code, appliquer limite=3 et la priorité de code.
    raise NotImplementedError("Exercice 5 : valider les arguments")

### Vérification ciblée — à lancer après l’implémentation

Le garde ci-dessous permet de lire/exécuter le support initial sans donner l’illusion que les exercices passent. Une cellule ignorée n’est pas un test réussi. Passer le drapeau à `True` une fois la fonction écrite ; enrichir ces cas avant extraction dans `tests/`.

In [ ]:
VALIDATION_IMPLEMENTEE = False
if VALIDATION_IMPLEMENTEE:
    invalides = ['{', '[]', 'null', {}, {"autre": "x"},
                 {"nom": ""}, {"nom": "   "}, {"code": 123},
                 {"nom": "lait", "limite": 0}, {"nom": "lait", "limite": 6},
                 {"nom": "lait", "limite": True}, {"nom": "lait", "limite": 1.5}]
    for entree in invalides:
        resultat = valider_arguments(entree)
        assert isinstance(resultat.get("erreur"), str) and resultat["erreur"], entree
    assert valider_arguments({"nom": "lait"})["limite"] == 3
    assert valider_arguments('{"code":"00123","limite":1}')["code"] == "00123"
    for limite in (1, 5):
        assert valider_arguments({"nom": "lait", "limite": limite})["limite"] == limite
    print("Cas de validation exécutés et réussis ; priorité code à vérifier avec la recherche.")
else:
    print("Vérifications non exécutées : exercice à compléter.")

## 6 — Examiner le catalogue réel avant de le brancher

Le dépôt expose `data/parquet/produits` via `src/storage/commun.py`. L’export `features_nutriscore_*` de `src/storage/export_features.py` est destiné au ML : il filtre les valeurs absentes et sa sélection ne contient pas de nom produit. Il ne faut donc pas l’utiliser aveuglément pour une recherche par nom ou une démonstration des manquants.

**À faire :** inspecter les colonnes et quelques lignes du catalogue TP12 en lecture seule. Si ce catalogue est indisponible, préparer les exercices de contrat hors ligne et différer la démonstration, ou constituer un extrait réel depuis Open Food Facts selon l’énoncé. Pas de relance ETL complète ni de modification de base pour cet exercice.

In [ ]:
INSPECTER_CATALOGUE = False  # Passer à True pour une lecture locale avec DuckDB installé.
if INSPECTER_CATALOGUE:
    import duckdb
    from contextlib import closing
    motif = RACINE / "data/parquet/produits/**/*.parquet"
    with closing(duckdb.connect(":memory:")) as connexion:
        relation = connexion.read_parquet(str(motif), hive_partitioning=True)
        print(list(zip(relation.columns, map(str, relation.types))))
        print(relation.limit(3).fetchall())
else:
    print("Inspection du catalogue non exécutée.")

### Construire le mapping — exercice

| Fiche outil | Colonne candidate, à confirmer | Unité |
|---|---|---|
| `code` | `code` | texte |
| `nom` | À relever dans le catalogue | texte |
| `nutriscore` | `nutriscore_grade` | A–E ou null |
| `nutriments.kcal` | `energy-kcal_100g` | kcal/100 g |
| `nutriments.lipides` | `fat_100g` | g/100 g |
| `nutriments.satures` | `saturated-fat_100g` | g/100 g |
| `nutriments.sucres` | `sugars_100g` | g/100 g |
| `nutriments.sel` | `salt_100g` | g/100 g |
| `nutriments.fibres` | `fiber_100g` | g/100 g |
| `nutriments.proteines` | `proteins_100g` | g/100 g |

Vérifier les unités du jeu retenu, notamment pour les boissons ; ne pas réétiqueter une mesure pour 100 ml en 100 g sans justification. Ne pas utiliser les valeurs imputées/scalées pour le ML comme des observations réelles.

Transformer les manquants en `None` (JSON `null`), conserver un vrai zéro et convertir les grades inconnus en `None`. `donnees_incompletes` vaut vrai si le grade ou l’un des sept nutriments manque. Un NaN ne doit pas fuiter dans le JSON : vérifier avec `json.dumps(fiche, allow_nan=False)`.

**Recherche à construire :** égalité exacte du code ; sinon recherche littérale d’une partie du nom après normalisation de la casse et des accents. Choisir un ordre stable. `total` compte tous les résultats **avant** troncature : plusieurs résultats restent `ambigu` même avec `limite=1`. Paramétrer le SQL ; si utilisation de LIKE, traiter `%` et `_` comme des caractères littéraux pour respecter le contrat.

In [ ]:
def chercher_produit(arguments):
    """Exposer le contrat TP18 ; dépendance au catalogue à définir explicitement."""
    # TODO : validation puis recherche ; ne pas lire le catalogue si arguments invalides.
    # TODO : adapter les fiches ; calculer total/statut avant la limite.
    raise NotImplementedError("Exercice 6 : recherche sur le catalogue")

### Plan de vérification de la recherche

Écrire d’abord des petites fixtures **synthétiques**, clairement étiquetées et réservées aux tests, puis passer aux produits réels pour la démonstration.

- Même recherche avec casse et accents différents ; sous-chaîne littérale.
- Code avec zéros de tête ; code prioritaire malgré un nom contradictoire.
- Zéro, un, plusieurs produits ; `limite=1` conserve l’ambiguïté et le total.
- Valeur réellement nulle conservée à 0 ; valeur manquante en null.
- Grade inconnu et nutriment absent : fiche incomplète ; JSON strict sérialisable.
- Argument invalide : résultat d’erreur, aucune interrogation du catalogue.

Extraire ensuite le code stable vers `src/assistant/` et les cas vers `tests/`. Cette organisation est une proposition, pas un module existant.

## 7 — Écrire les règles de réponse et de refus

**Exercice :** rédiger un prompt système bref. Inclure : périmètre alimentaire, refus du conseil médical avec orientation vers un professionnel, fiches comme seule source des chiffres, clarification si ambiguïté, absence explicite, indisponibilité actuelle de la substitution.

Les noms et textes du catalogue sont des **données non fiables** : une instruction glissée dans une fiche ne doit pas changer les règles de l’assistant. L’utilisateur ne peut pas autoriser un outil absent de la liste.

Pour expliquer un score, décrire les nutriments observés sans prétendre recalculer les points exacts si la méthode et les informations nécessaires ne sont pas disponibles.

**À vérifier :** une demande alimentaire normale, deux formulations de conseil médical, une question hors périmètre, une fiche contenant « ignore les règles », une valeur absente. Consigner les résultats réellement observés et les échecs.

In [ ]:
PROMPT_SYSTEME = """TODO : rédiger les règles de l’assistant, puis les confronter aux cas de la section 7."""

## 8 — Comprendre puis construire la boucle d’outils

Le modèle **demande** un outil ; votre programme décide si la demande est valide et l’exécute. Le modèle reçoit le résultat et formule sa réponse.

Pseudo-code à traduire après choix du client :

```text
initialiser messages, compteurs et échéance
pour chaque tour, jusqu’au budget fixé :
    appeler le modèle avec délai réseau et outils déclarés
    si réponse finale : vérifier la présence des preuves attendues puis terminer
    enregistrer le message assistant et ses demandes d’outils
    pour chaque demande, dans le budget global d’exécutions :
        vérifier le nom exact dans une liste autorisée
        valider les arguments sans eval
        exécuter chercher_produit ou produire un résultat d’erreur
        ajouter le résultat associé à la bonne demande dans l’historique
si budget épuisé : arrêter avec un message explicite
```

**Décisions à prendre :** budget de tours et budget total d’outils (plusieurs outils possibles par réponse), délai par appel, taille maximale des résultats, conduite après échec réseau ou sortie malformée. Un outil inconnu ne doit jamais devenir un appel dynamique arbitraire (`eval`, `exec`, résolution libre par nom).

Le format des messages et des identifiants dépend du client choisi. Le vérifier dans sa documentation officielle au moment de l’implémentation.

**Cas hors ligne :** outil inconnu, arguments invalides, plusieurs demandes, demandes répétées jusqu’au plafond, réponse sans outil, erreur du catalogue. Ces tests de contrôle ne remplacent pas l’appel sur un vrai modèle.

In [ ]:
MAX_TOURS = 4  # Valeurs pédagogiques proposées, à justifier.
MAX_APPELS_OUTILS = 6
DELAI_APPEL_SECONDES = 60

def dialoguer(question):
    """Implémenter une boucle bornée après avoir choisi le client réel."""
    # TODO : intégrer prompt, historique, liste autorisée et gestion des erreurs.
    # TODO : conserver appels, arguments et résultats pour l’audit des réponses.
    raise NotImplementedError("Exercice 8 : boucle outillée")

## 9 — Comparer les modes d’accès et préparer l’ADR

Ne pas choisir uniquement « parce que cela marche ». Comparer au moins les trois options de l’énoncé :

| Critère | Modèle local (ex. qwen3:8b / Ollama) | API hébergée en Europe | API commerciale |
|---|---|---|---|
| Qualité des appels et fidélité aux fiches | À tester | À tester | À tester |
| Latence et matériel | À mesurer | À mesurer | À mesurer |
| Coût et date de la source | Matériel / énergie, pas seulement tokens | À documenter | À documenter |
| Confidentialité et destination des données | À décrire | Région et conditions à vérifier | Conditions à vérifier |
| Limites pratiques | À relever | À relever | À relever |

Ne pas remplir de tarifs de mémoire. Citer les pages consultées et leur date. Ne jamais placer de clé dans une cellule, une sortie ou un commit ; utiliser une variable d’environnement. Ne pas afficher sa valeur.

Une mesure réelle sur l’option testée suffit pour commencer à étayer le choix ; les autres options peuvent rester documentaires, explicitement signalées. Distinguer premier appel à froid et appels suivants.

**À rédiger :** `docs/adr/0001-modele-et-mode-d-acces.md`, une page, statut **proposée** tant que le choix et sa mesure restent ouverts : Contexte, Options, Critères, Décision, Conséquences (dont une négative). Une décision acceptée ne se réécrit pas : une nouvelle ADR la remplace si nécessaire.

## 10 — Mesurer uniquement un vrai parcours

Cellule à activer après implémentation. `dialoguer` devra renvoyer `modele`, `reponse`, `appels_outils` et `resultats_outils`. La durée correspond au parcours entier, pas seulement à une requête réseau. Ajouter statut, nombre d’appels et éventuelle consommation de tokens selon ce que le fournisseur expose réellement.

In [ ]:
EXECUTER_MODELE_REEL = False
QUESTION_REELLE = "TODO : question sur un produit choisi dans le catalogue réel"
if EXECUTER_MODELE_REEL:
    from time import perf_counter
    debut = perf_counter()
    trace = dialoguer(QUESTION_REELLE)
    duree = perf_counter() - debut
    for cle in ("modele", "reponse", "appels_outils", "resultats_outils"):
        assert cle in trace, f"Trace manquante : {cle}"
    assert trace["appels_outils"], "Cette mesure doit inclure un appel outillé réel."
    print("Durée du parcours (s) :", round(duree, 3))
    print(json.dumps(trace, ensure_ascii=False, indent=2, allow_nan=False))
else:
    print("Aucun modèle appelé ; aucune latence mesurée.")

## 11 — Préparer les preuves de démonstration

Choisir les produits **après** inspection du catalogue. Pour chaque cas, conserver question, modèle exact, réponse, outils, arguments, fiches et durée. Relever chaque affirmation chiffrée de la réponse et la rattacher à un champ du résultat ; si un calcul est effectué, montrer ses opérandes et son calcul.

| Cas | Produit / question à choisir | Preuve à conserver | État initial |
|---|---|---|---|
| Produit connu | À compléter | Code et fiche retrouvée | Non exécuté |
| Comparaison ou explication | À compléter | Fiches, unités et limites de l’explication | Non exécuté |
| Introuvable ou valeur manquante | À compléter | Résultat vide ou champ null | Non exécuté |
| Refus médical ou hors périmètre | À compléter | Demande, refus, éventuels appels | Non exécuté |

Un script déterministe ou un modèle simulé peut tester l’orchestration, mais ne valide pas ces démonstrations. La partie 2 contiendra la démonstration exécutée si elle ne peut pas être réalisée ici.

## 12 — Préparer la première PR et la suite

Cette PR de préparation peut contenir l’énoncé et le présent notebook sans déclarer le TP terminé. Les exercices seront complétés progressivement ; ne cocher que ce qui a été réellement fait.

- [ ] Comportements et limites décrits.
- [ ] Architecture rédigée, avec état réel de chaque brique.
- [ ] Contrat, validation et cas de recherche vérifiés.
- [ ] Schéma réel du catalogue inspecté ; unités et manquants préservés.
- [ ] Prompt et boucle bornée implémentés et testés.
- [ ] Options comparées ; ADR proposée puis étayée par une mesure réelle.
- [ ] Démonstrations exécutées avec traces auditables, ici ou dans la partie 2.

**Partie 2 :** reprendre les cases ouvertes ; intégrer la substitution quand le TP16 sera prêt, sans changer le contrat `chercher_produit` ; compléter les preuves et accepter l’ADR après validation. RAG/API/application restent des étapes ultérieures, pas des conditions artificielles pour le premier appel outillé.

**Journal à compléter après la séance :** Fait / Problème / Suite, trois lignes dans `docs/journal.md`. Décrire le travail observé, pas l’intention.

La PR vise `SDMM27/NutriScope:develop` depuis le fork, selon `WORKFLOW_FORK_LOCAL.md`. Relecture par l’autre membre de l’équipe avant fusion.

### Sources

- [Énoncé original TP18](../../../docs/tps/README_tp_18.md).
- Cours mentionnés dans l’énoncé : `001_llm_fondamentaux.ipynb` et section 3 de `002_llm_echelle_embeddings_outils.ipynb` (à ouvrir dans le dossier de cours).
- [Ollama — appel d’outils](https://docs.ollama.com/capabilities/tool-calling).
- [Modèles d’ADR](https://adr.github.io/).
- [API Open Food Facts](https://openfoodfacts.github.io/openfoodfacts-server/api/).

**Données : © Open Food Facts contributors — ODbL 1.0**, [licence](https://opendatacommons.org/licenses/odbl/1-0/). Les fixtures synthétiques de tests ne sont pas des données OFF ni une démonstration réelle.